# Capstone --- Chapter 8: Planning, Decomposition and Replanning

Chapter~8 treats a plan as a work list: an ordered decomposition of a task into steps, subject to replanning when a step fails. The chapter distinguishes a free planner, which selects the next step at run time, from a fixed workflow, whose step order is decided in advance and enforced structurally. The capstone banking complaint agent takes the second position. Its plan is a fixed five-step workflow --- classify, extract, search policy, flag regulatory, draft --- and the ordering is not left to the model to rediscover on each call. It is encoded as a directed acyclic graph in the GMS banking store and enforced by a plausibility gate that rejects any transition the graph does not admit.

This companion reads that principle on the real capstone code. The workflow, the node order and the enabling relation are inspected offline; the store-backed gate is shown as reader-runnable, because scoring a transition loads the trained GMS store. The distinction the chapter draws --- a plan that is a work list versus a planner that improvises --- appears here as a fixed node sequence checked against a `has_enables` DAG.

## The plan as a fixed work list

The agent's plan is the module-level constant `_WORKFLOW_NODES`, an ordered list of the five workflow nodes. Two tool names differ from their node names, so the map `_TOOL_NODE_MAP` records the correspondence: the tools `classify_complaint` and `extract_facts` occupy the nodes `classify` and `extract`. Reading these two constants gives the entire plan; there is no run-time step selection to inspect.

In [1]:
from agentlab.capstone.complaint_agent import _WORKFLOW_NODES, _TOOL_NODE_MAP

print('workflow (in execution order):')
for i, node in enumerate(_WORKFLOW_NODES):
    print(f'  step {i}: {node}')
print()
print('tool -> node map:', _TOOL_NODE_MAP)

workflow (in execution order):
  step 0: classify
  step 1: extract
  step 2: search_policy
  step 3: flag_regulatory
  step 4: draft_response

tool -> node map: {'classify_complaint': 'classify', 'extract_facts': 'extract'}


## The plan encoded in the agent body

The fixed order is realized in `ComplaintAgent.propose_action`: the method branches on `state.step` and emits a specific `ToolCall` for each step, rather than asking the model which tool to call next. Step~0 classifies, step~1 extracts, step~2 searches policy, step~3 flags regulatory risk and step~4 drafts the reply. The source below shows the branch structure; the plan is literally the sequence of `if state.step == k` clauses.

In [2]:
import inspect
from agentlab.capstone.complaint_agent import ComplaintAgent

src = inspect.getsource(ComplaintAgent.propose_action)
# Show the step-dispatch skeleton: the lines that select a tool per step.
for line in src.splitlines():
    s = line.strip()
    if s.startswith('if state.step ==') or s.startswith('tool_name='):
        print(line)

        if state.step == 0:
        if state.step == 1:
        if state.step == 2:
                tool_name="search_policy",
        if state.step == 3:
                tool_name="flag_regulatory",
        if state.step == 4:
                tool_name="draft_response",


## The workflow DAG: the `has_enables` relation

The plan's ordering constraints are stored as triples over the relation `has_enables` in the GMS banking store. Each triple `(prev, has_enables, next)` asserts that `next` is an admissible successor of `prev`. The triples form a directed acyclic graph rooted at a synthetic `start` node. The edges below are read directly from the store's `triples.json`; they are the ground truth the plausibility gate scores against.

In [3]:
import json
from pathlib import Path

root = next((c for c in (Path('.'), Path('..'), Path('../code'), Path('code')) if (c / 'data').exists()), Path('.'))
store_path = root / 'data' / 'gms_banking_store'
triples = json.loads((store_path / 'triples.json').read_text())
enables = [t for t in triples if t[1] == 'has_enables']
print(f'{len(enables)} has_enables edges define the workflow DAG:')
for head, rel, tail in enables:
    print(f'  {head:16s} --{rel}--> {tail}')

7 has_enables edges define the workflow DAG:
  start            --has_enables--> classify
  classify         --has_enables--> extract
  extract          --has_enables--> search_policy
  search_policy    --has_enables--> flag_regulatory
  flag_regulatory  --has_enables--> draft_response
  flag_regulatory  --has_enables--> escalate
  draft_response   --has_enables--> escalate


The canonical spine `start -> classify -> extract -> search_policy -> flag_regulatory -> draft_response` is one path through this graph. The two additional edges into `escalate` --- from `flag_regulatory` and from `draft_response` --- encode the replanning branches of Chapter~8: a regulatory flag or an unsafe draft diverts the plan to escalation rather than continuing along the spine. The DAG therefore captures both the nominal work list and the sanctioned deviations from it.

## The previous node of a transition

The gate scores a transition `(prev_node, has_enables, tool_node)`. The context function `_prev_workflow_node` computes `prev_node` from the number of tool results recorded so far: before the first tool it returns `start`, and otherwise the node of the tool that ran last. The function reads only `len(state.tool_results)`, so its behavior can be exercised offline with a small stand-in object bearing that one attribute.

In [4]:
from types import SimpleNamespace
from agentlab.capstone.complaint_agent import _prev_workflow_node

# _prev_workflow_node reads only len(state.tool_results); a stand-in suffices
# for a structural demonstration (no store, no GPU).
for n in range(6):
    state = SimpleNamespace(tool_results=[{}] * n)
    prev = _prev_workflow_node(action=None, state=state)
    print(f'after {n} tool result(s): prev_node = {prev!r}')

after 0 tool result(s): prev_node = 'start'
after 1 tool result(s): prev_node = 'classify'
after 2 tool result(s): prev_node = 'extract'
after 3 tool result(s): prev_node = 'search_policy'
after 4 tool result(s): prev_node = 'flag_regulatory'
after 5 tool result(s): prev_node = 'draft_response'


## The plausibility gate enforces the plan

`build_complaint_harness` assembles three gates around the executor: a `SyntaxGate`, the policy engine as a gate and the `GMSPlausibilityGate`. The plausibility gate is the one that enforces the plan. It is constructed by `_build_gms_plausibility_gate`, which loads the trained store, reads the calibrated threshold from `calibration.json`, and binds the gate to the `has_enables` relation with `_prev_workflow_node` as its context function. The gate scores each proposed `(prev_node, has_enables, tool_node)` transition and denies any whose geodesic distance exceeds the threshold. The construction below loads the store; it is reader-runnable rather than executed in this build.

In [5]:
# Reader-runnable: this loads the trained GMS banking store (GPU if available).
# It is shown, not executed, in the notebook build.
from agentlab.capstone.complaint_agent import _build_gms_plausibility_gate

gate = _build_gms_plausibility_gate()
print('relation :', gate._relation)      # 'has_enables'
print('theta    :', gate._theta)         # calibrated operating point
print('on_missing:', gate._on_missing)   # behavior when the store is silent

knowlytix-core v1.1.0 licensed to customer=balimoon7@yahoo.com tier=developer expires=2026-12-21
By using this software you agree to https://knowlytix.ai/eula (suppress: set KNOWLYTIX_EULA_ACCEPTED=1 or touch /home/bsurjanto/.knowlytix/eula-accepted).


  GMS entities:  81
  GMS relations: 17
  GMS triples:   116
  Store loaded from /home/bsurjanto/projects/forgeloop/beyond-prompt-and-pray/code/data/gms_banking_store
  Entities:  81
  Relations: 17
  Triples:   116
  ENM:       15
  Documents: 1
relation : has_enables
theta    : 0.35
on_missing: allow


The calibrated threshold is not a default. It is read from the store's `calibration.json`, where it was fitted on a cohort of admissible and inadmissible transitions to a chosen false-allow ceiling. The cell below reads that record offline; the threshold is the operating point the gate applies to every transition.

In [6]:
import json
import sys
from pathlib import Path

root = next((c for c in (Path('.'), Path('..'), Path('../code'), Path('code')) if (c / 'data').exists()), Path('.'))
cal_path = root / 'data' / 'gms_banking_store' / 'calibration.json'

def _stale(path):
    if not path.exists():
        return True
    payload = json.loads(path.read_text())
    pg = payload.get('plausibility_gate', payload)
    return 'relation_set' not in pg

if _stale(cal_path):
    print('calibration.json missing or stale -- recalibrating ...')
    sys.path.insert(0, str(root / 'scripts'))
    import calibrate_gms_thresholds
    calibrate_gms_thresholds.main()

cal = json.loads(cal_path.read_text())
pg = cal['plausibility_gate'] if 'plausibility_gate' in cal else cal
print('relation set     :', pg['relation_set'])
print('threshold (theta):', pg['threshold'])
print('cohort size      :', pg['cohort_n'])
print('false-allow rate :', pg['false_allow_rate'])

calibration.json missing or stale -- recalibrating ...
  GMS entities:  81
  GMS relations: 17
  GMS triples:   116
  Store loaded from /home/bsurjanto/projects/forgeloop/beyond-prompt-and-pray/code/data/gms_banking_store
  Entities:  81
  Relations: 17
  Triples:   116
  ENM:       15
  Documents: 1
Sweeping plausibility threshold (theta) ...


  theta=0.40  acc=0.941  [CI 0.730, 0.990]  false_allow=0.100  false_deny=0.000  n=17
Sweeping contradiction threshold (tau_contra) ...
  tau_contra=1.00  acc=1.000  [CI 0.832, 1.000]  false_allow=0.000  false_deny=0.000  n=19
Wrote /home/bsurjanto/projects/forgeloop/beyond-prompt-and-pray/code/data/gms_banking_store/calibration.json
relation set     : ['has_enables']
threshold (theta): 0.4
cohort size      : 17
false-allow rate : 0.1


## Reading an out-of-order transition

The plan forbids skipping steps: `draft_response` may not follow `classify`, because no `has_enables` edge connects them. A free planner could propose such a jump; the fixed workflow does not, and the gate rejects it if it were proposed. The cell below shows the two transitions --- one on the spine, one a forbidden skip --- checked against the DAG edges read earlier, so the admissibility distinction is visible without loading the store. The trained gate scores these transitions geometrically rather than by exact edge lookup, but the DAG membership is what calibration teaches it to reproduce.

In [7]:
edge_set = {(h, t) for h, _r, t in enables}

candidates = [
    ('extract', 'search_policy'),      # on the spine: enabled
    ('flag_regulatory', 'escalate'),   # sanctioned replanning branch: enabled
    ('classify', 'draft_response'),    # forbidden skip: not enabled
    ('start', 'search_policy'),        # forbidden skip: not enabled
]
for prev, nxt in candidates:
    admissible = (prev, nxt) in edge_set
    verdict = 'ALLOW (in DAG)' if admissible else 'DENY (not in DAG)'
    print(f'{prev:16s} -> {nxt:16s} : {verdict}')

extract          -> search_policy    : ALLOW (in DAG)
flag_regulatory  -> escalate         : ALLOW (in DAG)
classify         -> draft_response   : DENY (not in DAG)
start            -> search_policy    : DENY (not in DAG)


## Summary and connections

The capstone realizes Chapter~8 by fixing the plan rather than planning at run time. The work list is the ordered `_WORKFLOW_NODES`; the ordering constraints and the sanctioned replanning branches are the `has_enables` DAG in the GMS banking store; and enforcement is the `GMSPlausibilityGate`, which scores each transition against that DAG at a calibrated threshold and denies what the plan does not admit. Decomposition is therefore a design-time decision made explicit in a graph, and replanning is confined to the escalation edges the graph sanctions. Chapter~8 develops the general contrast between free planners and fixed workflows and the conditions under which each is appropriate; the capstone chapter (Chapter~16) assembles this workflow together with the typed tools of Chapter~5 and the execution guard of Chapter~6 into the governed agent that runs end to end.